# Module 14 — Notebook 4 Solutions: Mini Project

> **These are complete solutions. Try the exercises yourself first!**

Come back here after you have attempted each step in `04_mini_project.ipynb`.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_contains, check_keys
Path('output').mkdir(exist_ok=True)
print("Setup complete.")

## Step 1 — Solution: Identify the Problems

The four categories of problems in `MESSY_CODE`:

- `magic_numbers` — `0.3`, `100`, `5` are unexplained literals
- `bad_names` — `d`, `x`, `y`, `z`, `s` say nothing about what they hold
- `no_docstring` — there are no functions, so there can't be docstrings
- `no_seed` — `random.sample()` runs without `random.seed()`, so results differ each run

In [ ]:
issues = [
    'magic_numbers',
    'bad_names',
    'no_docstring',
    'no_seed'
]

In [ ]:
check_type(issues, list, "issues is a list")
check_contains(issues, 'magic_numbers', "issues includes 'magic_numbers'")
check_contains(issues, 'bad_names', "issues includes 'bad_names'")
check_contains(issues, 'no_docstring', "issues includes 'no_docstring'")
check_contains(issues, 'no_seed', "issues includes 'no_seed'")

## Step 2 — Solution: Write the Clean Version

Key improvements:
- Module-level docstring explains the file's purpose
- `SEED`, `LONG_RESPONSE_THRESHOLD`, `SAMPLE_SIZE`, `FLAG_RATE_THRESHOLD` replace all magic numbers
- Three named functions replace the inline code — each has a docstring and type hints
- `random.seed(SEED)` is called once at module level before any random operation
- `if __name__ == '__main__':` means the module can be imported safely without running the analysis

In [ ]:
%%writefile clean_analysis.py
"""clean_analysis.py — Reproducible model output analysis."""
import json
import random
from pathlib import Path

SEED = 42
LONG_RESPONSE_THRESHOLD = 100
SAMPLE_SIZE = 5
FLAG_RATE_THRESHOLD = 0.3

random.seed(SEED)


def load_outputs(path: str) -> list:
    """Load model outputs from a JSON file."""
    with open(path) as f:
        return json.load(f)


def compute_flag_rate(outputs: list) -> float:
    """Return the fraction of outputs that are flagged."""
    flagged = [r for r in outputs if r['flagged']]
    return len(flagged) / len(outputs)


def get_long_responses(outputs: list, threshold: int = LONG_RESPONSE_THRESHOLD) -> list:
    """Return outputs whose response text exceeds threshold characters."""
    return [r for r in outputs if len(r['response']) > threshold]


if __name__ == '__main__':
    outputs = load_outputs('../../../data/synthetic/model_outputs.json')
    flag_rate = compute_flag_rate(outputs)
    if flag_rate > FLAG_RATE_THRESHOLD:
        print(f"High flag rate: {flag_rate:.2f}")
    long_responses = get_long_responses(outputs)
    print(f"Long responses: {len(long_responses)}")
    sample = random.sample(outputs, min(SAMPLE_SIZE, len(outputs)))
    for item in sample:
        print(item['model'], item['response'][:50])

In [ ]:
source = Path('clean_analysis.py').read_text()

In [ ]:
check_equal(Path('clean_analysis.py').exists(), True, "clean_analysis.py was created")
check_contains(source, 'SEED', "script defines SEED constant")
check_contains(source, 'def load_outputs', "script has load_outputs function")
check_contains(source, '"""', "script has docstrings")
check_contains(source, '__name__', "script has __name__ guard")

## Step 3 — Solution: Build the Reproducibility Report

The `reproducibility_report` dict records all the information needed to verify and re-run the analysis. In a real project this might be written to `output/metadata.json`.

In [ ]:
reproducibility_report = {
    'seeds_set': True,
    'script_path': 'clean_analysis.py',
    'data_path': '../../../data/synthetic/model_outputs.json',
    'constants_named': True,
    'has_docstrings': True
}

In [ ]:
check_keys(reproducibility_report, ['seeds_set', 'script_path', 'data_path', 'constants_named', 'has_docstrings'], "report has all required keys")
check_equal(reproducibility_report['seeds_set'], True, "seeds_set is True")
check_equal(Path(reproducibility_report['script_path']).exists(), True, "script_path points to a real file")